# Week 4 · Beyond Fine-Tuning: Distill, Merge, Quantize & Ship
### Build Custom AI — SarasAI Live Session 4

Your Analyst works. This week we answer the question every internal-AI project eventually faces:
**"great demo — now can it run fast, cheap, and private?"**

Three compression/consolidation tools, then the payoff — the full capstone pipeline assembled
and deployed:

| Technique | What it does | Training needed? |
|---|---|---|
| **Knowledge distillation** | Big *teacher* → small *student* | Yes (student) |
| **Model merging** | Combine several fine-tunes into one checkpoint | **No — not even a GPU** |
| **Quantization** | Shrink weights to 8/4-bit for cheap serving | No (post-training) |

> 📏 **The Evaluation Rule, final form:** compression is only acceptable if quality on the
> **Week-3 frozen test set** stays within a threshold *you set before compressing*.
> Today we also add the production axis: **latency (p50/p95), throughput, and $/1k requests.**

Primary sources: Distillation [arXiv:1503.02531](https://arxiv.org/abs/1503.02531) ·
TIES [arXiv:2306.01708](https://arxiv.org/abs/2306.01708) ·
GPTQ [arXiv:2210.17323](https://arxiv.org/abs/2210.17323) ·
AWQ [arXiv:2306.00978](https://arxiv.org/abs/2306.00978) ·
LLM.int8() [arXiv:2208.07339](https://arxiv.org/abs/2208.07339)

---
## 0 · Setup

In [ ]:
!pip install -q "transformers>=4.50" "trl>=0.17" "peft>=0.14" bitsandbytes datasets accelerate sentence-transformers gradio
# mergekit pins its own dependency ranges — install it separately so the resolver can't
# downgrade the stack above. (Safe to re-run.)
!pip install -q mergekit
# Colab pre-installs an old torchao that newer peft refuses to import alongside;
# we don't use it — removing it is safer than upgrading (its wheels are torch-coupled).
!pip uninstall -y -q torchao

In [ ]:
import torch, time, json, os

assert torch.cuda.is_available(), "This session needs a GPU."
print("Device:", torch.cuda.get_device_name(0))

BF16_OK = torch.cuda.is_bf16_supported()          # False on T4 (pre-Ampere)
DTYPE = torch.bfloat16 if BF16_OK else torch.float16
print("Using dtype:", DTYPE)

---
## 0.5 · Self-contained Analyst bootstrap  `(train-or-load, ~4 min first run)`

**This notebook has no dependencies on previous weeks.** It trains a quick QLoRA Analyst right
here (a compressed replay of Week 3: template data → LoRA → save), so everyone starts from the
same place — whether or not you attended Week 3. Already ran this cell before? It loads the
saved adapter instantly.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import random, itertools

BASE_ID = "Qwen/Qwen2.5-1.5B-Instruct"
tok = AutoTokenizer.from_pretrained(BASE_ID)
ADAPTER_DIR = "analyst-adapter"

def make_input(defect_note, evidence):
    return (f"Defect note from visual inspection:\n{defect_note}\n\n"
            f"Retrieved evidence:\n{evidence}\n\nWrite a root cause analysis report.")

# deterministic mini-dataset (5 products x 8 defects, 3 sentence templates)
random.seed(42)
_P = ["KT-2000 kettle", "TS-400 toaster", "BL-900 blender", "MW-70 microwave", "SG-5 steam iron"]
_D = [("cracked lid hinge","HIGH"),("discolored housing","LOW"),("lever sticking","MEDIUM"),
      ("seal degradation","MEDIUM"),("blade wobble","HIGH"),("loose power cord","HIGH"),
      ("faded print","LOW"),("rattling component","MEDIUM")]
_T = ["{d} observed on {n} units of {p}, batch B-{b}.",
      "QC flagged {n} {p} units (batch B-{b}) for {d}.",
      "Inspection of batch B-{b} found {d} on {n} units of the {p}."]
rows = []
combos = list(itertools.product(_P, _D)); random.shuffle(combos)
for i, (p, (d, sev)) in enumerate((combos * 2)[:52]):
    note = _T[i % 3].format(d=d.capitalize(), n=random.randint(2, 9), p=p, b=1000 + i)
    ev = (f"- review_{i:03d}: customer reports '{d}' soon after purchase\n"
          f"- service log: reports {'rising' if sev != 'LOW' else 'stable'} this quarter")
    rep = (f"## ROOT CAUSE ANALYSIS\n**Product:** {p}\n**Defect:** {d}\n**Severity:** {sev}\n"
           f"**Evidence:** Customer review corroborates the inspection note.\n"
           f"**Probable cause:** Supplier lot variation.\n"
           f"**Recommended action:** Audit batch B-{1000+i} for {d}.")
    rows.append({"input": make_input(note, ev), "output": rep})

test_rows, train_rows = rows[:12], rows[12:]          # 12 frozen rows for the quality gate
print(f"train {len(train_rows)} | test {len(test_rows)} (frozen 🔒)")

In [ ]:
if os.path.exists(ADAPTER_DIR):
    print("adapter found on disk — skipping training")
else:
    from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
    from trl import SFTTrainer, SFTConfig
    from datasets import Dataset
    bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                             bnb_4bit_compute_dtype=DTYPE, bnb_4bit_use_double_quant=True)
    _base4 = AutoModelForCausalLM.from_pretrained(BASE_ID, quantization_config=bnb, device_map="auto")
    _m = get_peft_model(prepare_model_for_kbit_training(_base4),
        LoraConfig(r=8, lora_alpha=16, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
                   target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"]))
    _ds = Dataset.from_list([{"messages": [{"role": "user", "content": r["input"]},
                                           {"role": "assistant", "content": r["output"]}]} for r in train_rows])
    SFTTrainer(model=_m, train_dataset=_ds, processing_class=tok,
        args=SFTConfig(output_dir=ADAPTER_DIR, num_train_epochs=2, per_device_train_batch_size=2,
            gradient_accumulation_steps=4, learning_rate=2e-4, logging_steps=5,
            bf16=BF16_OK, fp16=not BF16_OK, optim="paged_adamw_8bit",
            max_length=1024, report_to="none")).train()
    _m.save_pretrained(ADAPTER_DIR)
    del _m, _base4; torch.cuda.empty_cache()
    print("bootstrap Analyst trained + saved -> ./" + ADAPTER_DIR)

---
## 1 · Knowledge distillation: the teacher–student idea

Hinton's 2015 insight: a trained model's *full output distribution* — not just its top answer —
encodes how it "thinks" (crack: 0.7, fracture: 0.25, banana: 0.0001…). Training a small student
to match those **soft targets** transfers far more signal per example than hard labels do.

Two flavors you'll meet in practice:

1. **Black-box / data distillation** — teacher *generates outputs*, student does SFT on them.
   **You already did this in Week 3** — your synthetic dataset was distillation in disguise.
2. **White-box / logit distillation** — student matches the teacher's token distributions
   directly (needs open weights). TRL packages this as `GKDTrainer`.

The white-box recipe in five lines of intuition — a KL-divergence loss between distributions,
softened by temperature T:

In [ ]:
import torch.nn.functional as F

def distill_loss(student_logits, teacher_logits, T=2.0):
    """The classic KD loss (Hinton 2015): match softened distributions, not argmax labels."""
    return F.kl_div(
        F.log_softmax(student_logits / T, dim=-1),
        F.softmax(teacher_logits / T, dim=-1),
        reduction="batchmean",
    ) * T * T

# toy demonstration: a 4-token vocabulary
teacher = torch.tensor([[5.0, 3.0, 1.0, -2.0]])      # confident but nuanced
student = torch.tensor([[2.0, 2.0, 2.0,  2.0]])      # clueless (uniform)
print(f"loss (clueless student): {distill_loss(student, teacher):.3f}")
print(f"loss (matched student):  {distill_loss(teacher.clone(), teacher):.3f}")

In TRL the full white-box pipeline is a drop-in trainer — same shape as Week 3's SFT
(full-scale recipe — we run a session-scale version for real right after the merge step below):

```python
# GKD moved between trl versions — resolve it wherever this trl keeps it
try:
    from trl import GKDConfig, GKDTrainer                      # trl <= 0.2x (top-level)
except ImportError:
    try:
        from trl.experimental.gkd import GKDConfig, GKDTrainer # newer trl (experimental namespace)
    except ImportError:
        from trl.trainer.gkd_trainer import GKDConfig, GKDTrainer  # last-resort internal path          # Generalized Knowledge Distillation

trainer = GKDTrainer(
    model="Qwen/Qwen2.5-0.5B-Instruct",        # small student
    teacher_model="Qwen/Qwen2.5-1.5B-Instruct",# your fine-tuned Analyst as teacher
    args=GKDConfig(output_dir="analyst-student", lmbda=0.5, beta=0.5),
    train_dataset=train_ds,                    # same chat format as Week 3
    processing_class=tok,
)
```

**When to reach for distillation:** you need a *smaller* model than your fine-tune (edge,
high-QPS), you have (or can generate) plenty of task inputs, and you can afford one big
training run. Distill from your *fine-tuned* model — it's the best teacher of your format.

---
## 2 · Model merging: consolidation without training

Different problem: you now have **several** LoRA fine-tunes of the *same base* — the Analyst,
maybe a summarizer, a colleague's Q&A tune. Serving three checkpoints costs 3× memory.

**Merging** arithmetically combines them into one checkpoint. No data, no gradients, no GPU.
- **SLERP** — spherical interpolation between two models.
- **TIES** (Yadav 2023) — trims tiny deltas, resolves sign conflicts, merges. Best for 2+ models.
- **DARE** — random-drops deltas then rescales; often stacked with TIES.

> ⚠️ **What merging is NOT** (adversarially fact-checked in our curriculum research): it does
> **not** create abilities beyond its parents. It *consolidates* — think "one employee with two
> skills", not "a smarter employee". Benefits also grow with model scale; at 1.5B expect
> workmanlike results, not magic. **The merged model must re-pass each parent's eval** — that's
> the acceptance test.

First, the simplest possible merge — fold our LoRA adapter into the base weights (you'll need
this for quantization anyway):

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

base = AutoModelForCausalLM.from_pretrained(BASE_ID, torch_dtype=DTYPE,
                                            device_map="auto")

# load Week 3's adapter and merge it into the weights → a single standalone checkpoint
analyst = PeftModel.from_pretrained(base, ADAPTER_DIR)   # ← the bootstrap adapter from §0.5
analyst = analyst.merge_and_unload()
analyst.save_pretrained("analyst-merged")
tok.save_pretrained("analyst-merged")
print("Adapter folded into base → ./analyst-merged (standalone model)")

> 🔍 **Honest fine print:** the adapter was *trained* against NF4-quantized base weights, but
> we just merged it into half-precision weights — the combined model is therefore slightly
> different from what you evaluated in Week 3. Usually negligible; the frozen-set re-score
> below is exactly what catches it when it isn't.

### Now let's actually run it — small-scale white-box distillation on the T4

The sketch above is the full-scale recipe. Here we run the real thing at session scale:
**teacher = your fine-tuned Analyst** (4-bit base + Week-3 adapter), **student =
Qwen2.5-0.5B** — 3× smaller. Two steps, exactly the two flavors: (1) the teacher writes
reports for a small prompt set (black-box data), (2) `GKDTrainer` additionally matches the
teacher's **token distributions** while training the student on them (white-box). ~5–8 min;
the production version is the same code with 10–100× the data.

In [ ]:
# distillation data: 16 fresh prompts (NOT the frozen test set — that stays untouched),
# with the teacher's own outputs as targets
import random
random.seed(7)
_prods = ["KT-2000 kettle", "TS-400 toaster", "BL-900 blender", "MW-70 microwave"]
_defs = ["cracked lid hinge", "lever sticking", "seal degradation", "blade wobble"]
distill_prompts = [make_input(
    f"{d.capitalize()} observed on {random.randint(2,9)} units of {p}, batch D-{5000+i}.",
    f"- review_d{i:02d}: customer reports '{d}' soon after purchase\n- spec: component rated for standard household use")
    for i, (p, d) in enumerate((random.sample([(p, d) for p in _prods for d in _defs], 16)))]

if os.path.exists("distill_pairs.json"):
    distill_pairs = json.load(open("distill_pairs.json"))
else:
    def _chat_t(text, max_new_tokens=250):          # local helper — teacher generation
        inputs = tok.apply_chat_template([{"role": "user", "content": text}],
            add_generation_prompt=True, return_dict=True, return_tensors="pt").to(analyst.device)
        out = analyst.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False,
                               pad_token_id=tok.eos_token_id)
        return tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()
    distill_pairs = [{"messages": [{"role": "user", "content": x},
                                   {"role": "assistant", "content": _chat_t(x)}]}
                     for x in distill_prompts]
    json.dump(distill_pairs, open("distill_pairs.json", "w"))
print(len(distill_pairs), "teacher-written training pairs (the black-box half)")

In [ ]:
# the white-box half: GKDTrainer — student trains on the pairs AND matches teacher logits
# GKD moved between trl versions — resolve it wherever this trl keeps it
try:
    from trl import GKDConfig, GKDTrainer                      # trl <= 0.2x (top-level)
except ImportError:
    try:
        from trl.experimental.gkd import GKDConfig, GKDTrainer # newer trl (experimental namespace)
    except ImportError:
        from trl.trainer.gkd_trainer import GKDConfig, GKDTrainer  # last-resort internal path
from datasets import Dataset

STUDENT_ID = "Qwen/Qwen2.5-0.5B-Instruct"
student_tok = AutoTokenizer.from_pretrained(STUDENT_ID)

if os.path.exists("analyst-student"):
    student = AutoModelForCausalLM.from_pretrained("analyst-student", torch_dtype=DTYPE, device_map="auto")
    print("loaded distilled student from disk")
else:
    student = AutoModelForCausalLM.from_pretrained(STUDENT_ID, torch_dtype=DTYPE, device_map="auto")
    gkd_cfg = GKDConfig(output_dir="analyst-student", num_train_epochs=2,
        per_device_train_batch_size=1, gradient_accumulation_steps=2,
        learning_rate=2e-5, lmbda=0.5, beta=0.5, max_new_tokens=128,
        logging_steps=4, bf16=BF16_OK, fp16=not BF16_OK, report_to="none")
    GKDTrainer(model=student, teacher_model=analyst, args=gkd_cfg,
               train_dataset=Dataset.from_list(distill_pairs),
               processing_class=student_tok).train()
    student.save_pretrained("analyst-student"); student_tok.save_pretrained("analyst-student")
student.eval()
print(f"student footprint: {student.get_memory_footprint()/1e9:.2f} GB  (teacher: {analyst.get_memory_footprint()/1e9:.2f} GB)")

In [ ]:
# did the knowledge transfer? Before/after on 6 unseen inputs (quick, 0.5B is fast)
def chat_student(m, text, max_new_tokens=250):
    inputs = student_tok.apply_chat_template([{"role": "user", "content": text}],
        add_generation_prompt=True, return_dict=True, return_tensors="pt").to(m.device)
    out = m.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False,
                     pad_token_id=student_tok.eos_token_id)
    return student_tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

vanilla = AutoModelForCausalLM.from_pretrained(STUDENT_ID, torch_dtype=DTYPE, device_map="auto")
# 6 UNSEEN prompts from the same generator (the frozen test set stays untouched for the gate)
sample = [make_input(f"{d.capitalize()} observed on 3 units of {p}, batch E-{6000+i}.",
                     f"- review_e{i:02d}: customer reports '{d}'")
          for i, (p, d) in enumerate([("KT-2000 kettle","faded print"),("TS-400 toaster","loose power cord"),
                                      ("BL-900 blender","rattling component"),("MW-70 microwave","thermostat drift"),
                                      ("KT-2000 kettle","discolored housing"),("BL-900 blender","cracked base")])]
rate_v = sum(format_ok(chat_student(vanilla, x)) for x in sample) / len(sample)
rate_s = sum(format_ok(chat_student(student, x)) for x in sample) / len(sample)
print(f"0.5B format adherence — vanilla: {rate_v:.0%}   distilled: {rate_s:.0%}")
del vanilla, student
torch.cuda.empty_cache()
print("(cleaned up — memory back for the quantization sections)")

**Read the result honestly:** 16 pairs and 2 epochs won't make a 0.5B match your 1.5B — but
the format-adherence jump is real transfer, measured on unseen inputs. Scale the same code to
your Week-3 training set (or 10× it) and the gap closes further. That's the distillation
deal: **one big training run buys a model 3× cheaper to serve, forever** — and your
fine-tuned Analyst is the best teacher of your format that exists.

And the multi-model case with **mergekit** — declarative YAML in, merged model out.
This runs on CPU; it's pure tensor arithmetic:

In [ ]:
merge_config = """
models:
  - model: ./analyst-merged            # our fine-tuned Analyst
    parameters: {weight: 0.6, density: 0.6}
  - model: Qwen/Qwen2.5-1.5B-Instruct  # NOTE: a placeholder — in real life this is a SECOND
    parameters: {weight: 0.4, density: 0.6}   # fine-tune (a summarizer, a colleague's model)
merge_method: ties
base_model: Qwen/Qwen2.5-1.5B-Instruct
dtype: float16
"""

with open("merge.yaml", "w") as f:
    f.write(merge_config)

# uncomment to run (~2 min, CPU-friendly):
# !mergekit-yaml merge.yaml ./analyst-ties --allow-crimes
print("mergekit config written — density=0.6 keeps the top 60% of deltas (TIES trimming)")

**The acceptance test is non-negotiable:** a merged model must *re-pass each parent's eval*
before it replaces anything. With our placeholder second parent, TIES here just scales the
Analyst's deltas — but the checking code is the same for a real merge:

```python
# merged = AutoModelForCausalLM.from_pretrained("./analyst-ties", torch_dtype=DTYPE, device_map="auto")
# print("merged format adherence:", rate([chat(merged, x) for x in test_inputs]))   # parent 1's eval
# print("merged summarization score:", ...)                                          # parent 2's eval
```

---
## 3 · Quantization: the highest-ROI compression there is

The empirical headline from three papers (GPTQ, AWQ, LLM.int8): **weights tolerate 4–8-bit
precision with minor quality loss** — AWQ's refinement being that ~1% of *salient* channels
matter far more than the rest and get protected.

Three formats you'll actually encounter:

| Format | Tooling | Where it shines |
|---|---|---|
| **bitsandbytes** 8/4-bit | `load_in_*bit=True` | Zero-effort GPU serving & QLoRA training |
| **GPTQ / AWQ** | `optimum`, `autoawq` | Calibrated 4-bit for GPU inference engines |
| **GGUF** | `llama.cpp` / Ollama | **CPU/edge** — the "runs on a laptop" format |

Measure, don't trust — memory first:

In [ ]:
from transformers import BitsAndBytesConfig

def footprint(model):
    return f"{model.get_memory_footprint()/1e9:.2f} GB"

print("half-precision Analyst:", footprint(analyst))

analyst_4bit = AutoModelForCausalLM.from_pretrained(
    "analyst-merged",
    quantization_config=BitsAndBytesConfig(load_in_4bit=True,
                                           bnb_4bit_quant_type="nf4",
                                           bnb_4bit_compute_dtype=DTYPE),
    device_map="auto",
)
print("4-bit Analyst:", footprint(analyst_4bit))

~4× smaller. For CPU/edge deployment you'd additionally export **GGUF** (one-time, ~10 min —
note the build step: `llama-quantize` is a binary that must be compiled first):

```bash
git clone --depth 1 https://github.com/ggml-org/llama.cpp
pip install -r llama.cpp/requirements.txt

# 1. convert HF checkpoint → GGUF at fp16 (a Python script — no build needed)
python llama.cpp/convert_hf_to_gguf.py ./analyst-merged --outfile analyst-f16.gguf

# 2. build the quantizer, then quantize to q4_k_m
cmake -B llama.cpp/build llama.cpp
cmake --build llama.cpp/build --target llama-quantize -j
./llama.cpp/build/bin/llama-quantize analyst-f16.gguf analyst-q4_k_m.gguf q4_k_m
```

`q4_k_m` is the community-standard "minor loss, big savings" preset — then `ollama` or
`llama-server` runs the Analyst on any laptop. **Private by construction.**

---
## 4 · The gate: quality on the frozen Week-3 test set

Set the threshold **before** looking at results. We commit, in writing, now:
**neither format adherence nor the content score may drop more than 5 points vs. the
half-precision model.** Gating on format alone would be a mistake — a compressed model can
keep every header intact while the analysis inside degrades. Then re-run the Week-3 eval:

In [ ]:
import re, os

# test_rows: the 12 frozen rows from the §0.5 bootstrap — untouched since.
print(f"frozen test set: {len(test_rows)} rows 🔒")

REQUIRED = ["## ROOT CAUSE ANALYSIS", "**Product:**", "**Defect:**", "**Severity:**",
            "**Evidence:**", "**Probable cause:**", "**Recommended action:**"]

def format_ok(t):
    return all(h in t for h in REQUIRED) and bool(
        re.search(r"\*\*Severity:\*\*\s*(LOW|MEDIUM|HIGH)", t))

def chat(model, text, max_new_tokens=350):
    msgs = [{"role": "user", "content": text}]
    inputs = tok.apply_chat_template(msgs, add_generation_prompt=True,
                                     return_dict=True, return_tensors="pt").to(model.device)
    out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False,
                         pad_token_id=tok.eos_token_id)
    return tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

Two metrics, mirroring Week 3's eval: **format adherence** (the mechanical secondary) and a
**content score** — embedding similarity between each generated report and Week 3's reference
output. The content score is deterministic (no judge to bias) and catches the failure format
checking can't: headers intact, content degraded.

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

embedder = SentenceTransformer("BAAI/bge-small-en-v1.5")
refs = embedder.encode([r["output"] for r in test_rows], normalize_embeddings=True)

def content_score(outputs):
    gen = embedder.encode(outputs, normalize_embeddings=True)
    return float((gen * refs).sum(axis=1).mean())      # mean cosine vs reference reports

test_inputs = [r["input"] for r in test_rows]

out_half = [chat(analyst, x)      for x in test_inputs]     # half-precision baseline
out_4bit = [chat(analyst_4bit, x) for x in test_inputs]     # compressed candidate

In [ ]:
rate = lambda outs: sum(map(format_ok, outs)) / len(outs)

fmt_drop     = (rate(out_half) - rate(out_4bit)) * 100
content_drop = (content_score(out_half) - content_score(out_4bit)) * 100

print(f"format adherence — half: {rate(out_half):.0%}   4-bit: {rate(out_4bit):.0%}   Δ {fmt_drop:+.1f} pts")
print(f"content score    — half: {content_score(out_half):.3f}  4-bit: {content_score(out_4bit):.3f}   Δ {content_drop:+.1f} pts")

ok = fmt_drop <= 5 and content_drop <= 5          # BOTH must hold — committed in §4's markdown
print("verdict:", "✅ within threshold — ship it" if ok else "❌ REJECT — stay at 8-bit or re-calibrate")

That ✅/❌ line is the deliverable pattern: **a pre-committed threshold and a pass/fail
verdict**, not "it seems fine".

---
## 5 · The production axis: latency, throughput, cost

Quality survived — but the *reason* we quantized is economics. Measure p50/p95 latency
(p95 is what your users feel; SLOs are written against it) and tokens/sec:

In [ ]:
def bench(model, prompt, n_runs=10, max_new_tokens=150):
    times, tokens = [], []
    for _ in range(n_runs):
        t0 = time.perf_counter()
        out_text = chat(model, prompt, max_new_tokens=max_new_tokens)
        times.append(time.perf_counter() - t0)
        tokens.append(len(tok.encode(out_text)))         # count what was ACTUALLY generated
    t = np.array(times)
    return {"p50_s": round(float(np.percentile(t, 50)), 2),
            "p95_s": round(float(np.percentile(t, 95)), 2),
            "tok_per_s": round(float(np.mean(tokens) / np.mean(t)), 1)}

prompt = test_inputs[0]
stats_half = bench(analyst, prompt)
stats_4bit = bench(analyst_4bit, prompt)
print("half :", stats_half)
print("4-bit:", stats_4bit)

> ⚠️ **Don't be surprised if 4-bit is *not faster* here.** bitsandbytes 4-bit optimizes
> *memory*, and dequantization adds compute — on a T4 it can even be slightly slower per token.
> The speed win comes from what the freed memory *buys you*: bigger batches, longer contexts,
> cheaper GPUs, or CPU serving via GGUF. **Quantization gains are hardware- and
> engine-dependent** (this was one of our verified research findings) — which is why you
> benchmark on the target stack, never extrapolate.

Cost per 1k requests, back-of-envelope — the number your CFO actually asks for:

In [ ]:
GPU_PRICE_PER_H = 0.40                      # e.g. RunPod T4/A4000-class; plug in your real price

for name, stats in [("half", stats_half), ("4-bit", stats_4bit)]:   # reuse §5's measurements
    cost_1k = GPU_PRICE_PER_H / 3600 * stats["p50_s"] * 1000
    print(f"{name:5}  p50={stats['p50_s']}s  →  ${cost_1k:.2f} per 1k reports")
# caveat: sequential requests — a batching engine (§7) divides this by the batch size

---
## 6 · The payoff: assembling the capstone pipeline

Everything from four weeks, wired together:

```
product photo ──► [Week 1] VLM inspector ──► defect note (JSON)
                                                  │
customer question / batch id                      ▼
        └────────► [Week 2] RAG retrieval ──► evidence chunks
                                                  │
                                                  ▼
                   [Week 3+4] quantized Analyst ──► ROOT CAUSE ANALYSIS report
```

Each stage was **evaluated independently** (defect P/R, recall@k + faithfulness, format +
win-rate). That is what makes the composite debuggable: when a report is wrong, you check the
stage evals and know *where* it broke.

The cell below wires the stages together. `vlm_inspect` and `rag_retrieve` are **interface
stubs** — in your capstone you replace their bodies with your Week 1 `inspect()` and Week 2
`retrieve()` (paste the cells or import from saved modules). The stubs keep this notebook
runnable standalone so we can demo the *composition* today:

In [ ]:
def vlm_inspect(image):
    """STUB — replace with Week 1's inspect(): VLM → {"verdict", "defect_type", "description"}."""
    return {"verdict": "DEFECTIVE", "defect_type": "crack",
            "description": "hairline crack near the lid seal"}

def rag_retrieve(query, k=4):
    """STUB — replace with Week 2's retrieve(): query → [{"source", "text"}, ...]."""
    return [{"source": "review_041", "text": "lid cracked after one week of normal use"},
            {"source": "spec_kettle", "text": "lid assembly is polypropylene, part PP-114"}]

def make_input(defect_note, evidence):
    """Same contract as Week 3 — this string format is what the Analyst was trained on."""
    return (f"Defect note from visual inspection:\n{defect_note}\n\n"
            f"Retrieved evidence:\n{evidence}\n\nWrite a root cause analysis report.")

With the interfaces pinned down, the pipeline itself is ten lines — this is the payoff of
designing each week's component around a clean contract:

In [ ]:
def analyze_product(image, question="What defects are visible?"):
    """The full capstone pipeline in one function."""
    # Stage 1 — VLM inspection (Week 1)
    note = vlm_inspect(image)                  # {"verdict": ..., "defect_type": ..., "description": ...}
    if note.get("verdict") == "GOOD":
        return {"note": note, "report": "No defect found — no analysis needed."}

    # Stage 2 — retrieve evidence about this defect (Week 2)
    hits = rag_retrieve(f"{note.get('defect_type', '')} {note.get('description', '')}", k=4)
    evidence = "\n".join(f"- {h['source']}: {h['text'][:120]}" for h in hits)

    # Stage 3 — quantized Analyst writes the report (Weeks 3+4)
    note_text = f"{note.get('defect_type', 'defect')}: {note.get('description', '')}"  # prose, like training
    report = chat(analyst_4bit, make_input(note_text, evidence))
    return {"note": note, "evidence": evidence, "report": report}

result = analyze_product(None)                 # stub inspect ignores the image — end-to-end smoke test
print(result["report"][:400])

And a shareable UI in ~15 lines with Gradio — push this exact app to a **Hugging Face Space**
and the pipeline has a URL:

In [ ]:
import gradio as gr

def ui_fn(image, question):
    result = analyze_product(image, question)
    return json.dumps(result["note"], indent=2), result["report"]

demo = gr.Interface(
    fn=ui_fn,
    inputs=[gr.Image(type="pil", label="Product photo"),
            gr.Textbox(label="Question", value="What defects are visible?")],
    outputs=[gr.Textbox(label="Inspection note (VLM)"),
             gr.Textbox(label="Root cause analysis (Analyst)")],
    title="Product Risk Pipeline — VLM ► RAG ► Analyst SLM",
)
# demo.launch(share=True)     # uncomment in the live session

---
## 7 · Production considerations: real serving infrastructure

A notebook calls `model.generate()` — one request at a time, one user. Production serving is a
different discipline, and there are purpose-built engines so you don't build it yourself:

| Layer | Notebook | Production |
|---|---|---|
| Engine | `transformers.generate()` | **vLLM** or **TGI** — continuous batching, paged attention: 5–20× throughput on the same GPU |
| CPU/edge | — | **GGUF + llama.cpp / Ollama** (your q4_k_m artifact drops straight in) |
| Scaling | one GPU | autoscaler + queue, sized against **p95 SLO** |
| Rollout | overwrite the model | shadow deploys; frozen-eval gate **in CI** before any model swap |
| Watching | print() | p50/p95, tokens/sec, GPU util, $/1k, **and the quality evals** on a dashboard |

One-liner to remember: `vllm serve ./analyst-merged` — your merged checkpoint is already
compatible. **Quantization is a serving decision**: it sets your memory, batch capacity, and
cost ceiling, so it's chosen against the SLO, not for the demo.

---
## 8 · Your assignments

**Ungraded warm-up:** export the Analyst to GGUF q4_k_m (commands in §3), run it with Ollama
on your own laptop, and re-run the format-adherence eval there. Report the number and your
tokens/sec — welcome to edge deployment.

**Graded — Capstone Increment 4 (final):** deliver the optimized, integrated pipeline. Submit:
(1) quantized Analyst + the **pre-committed quality threshold** and pass/fail table on the
Week-3 frozen set, (2) p50/p95 latency + tokens/sec + $/1k on your serving setup,
(3) the integrated pipeline (notebook or Space) processing ≥3 example products end-to-end,
(4) deployment note: chosen serving stack and why.

**Week 5 — Capstone Delivery:** presentation + evaluation appendix consolidating every held-out
metric from all four increments (W1 defect P/R → W2 recall@k/faithfulness → W3 format/win-rate →
W4 post-compression delta + latency/cost). *That appendix — a system you can prove works — is
the single most portfolio-worthy artifact of this program.*

🎓 *You now own the full custom-AI toolbox: prompt → retrieve → fine-tune → compress → ship —
with an evaluation spine holding it together.*